In [41]:
import os
import sys

import numpy as np
import pandas as pd
import torch

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [42]:
device.type

'cpu'

In [43]:
%load_ext autoreload
%autoreload 2

from drGAT import drGAT

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
train_data = pd.read_csv("../CTRP_data/train.csv")
val_data = pd.read_csv("../CTRP_data/val.csv")
test_data = pd.read_csv("../CTRP_data/test.csv")

In [45]:
train_data.head()

,DRUG_NAME,CELL_LINE_NAME
0,BRD-K27986637,RERFLCAI
1,BRD-K09344309,NCIH1666
2,quizartinib,OVTOKO
3,BRD-K99584050,SNU878
4,ML210,YD15


In [46]:
idxs = np.load("../CTRP_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [47]:
edge_index = np.load("../CTRP_data/edge_idxs.npy")
edge_attr = np.load("../CTRP_data/edge_attr.npy")

In [48]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["DRUG_NAME"]]
    X["Cell"] = [converter[(i)] for i in X["CELL_LINE_NAME"]]
    return X

In [49]:
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [50]:
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)

In [51]:
edge_attr = torch.tensor(edge_attr).float()

In [52]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [53]:
train_labels = np.load("../CTRP_data/train_labels.npy")
val_labels = np.load("../CTRP_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

## Get feature matrix

In [54]:
drug = pd.read_csv("../CTRP_data/drug_sim.csv", index_col=0)
cell = pd.read_csv("../CTRP_data/cell_sim.csv", index_col=0)
gene = pd.read_csv("../CTRP_data/gene_sim.csv", index_col=0)

In [55]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [56]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]
data

[tensor([[1.0000, 0.7189, 0.7661,  ..., 0.7661, 0.5431, 0.7898],
         [0.7189, 1.0000, 0.7122,  ..., 0.7391, 0.5431, 0.7560],
         [0.7661, 0.7122, 1.0000,  ..., 0.7661, 0.5694, 0.7560],
         ...,
         [0.7661, 0.7391, 0.7661,  ..., 1.0000, 0.6887, 0.7898],
         [0.5431, 0.5431, 0.5694,  ..., 0.6887, 1.0000, 0.5727],
         [0.7898, 0.7560, 0.7560,  ..., 0.7898, 0.5727, 1.0000]]),
 tensor([[1.0000, 0.8669, 0.8635,  ..., 0.6272, 0.7101, 0.6883],
         [0.8669, 1.0000, 0.7880,  ..., 0.6278, 0.7546, 0.6731],
         [0.8635, 0.7880, 1.0000,  ..., 0.5572, 0.6405, 0.6263],
         ...,
         [0.6272, 0.6278, 0.5572,  ..., 1.0000, 0.6355, 0.8001],
         [0.7101, 0.7546, 0.6405,  ..., 0.6355, 1.0000, 0.7164],
         [0.6883, 0.6731, 0.6263,  ..., 0.8001, 0.7164, 1.0000]]),
 tensor([[1.0000, 0.0955, 0.1206,  ..., 0.1463, 0.1351, 0.1001],
         [0.0955, 1.0000, 0.0845,  ..., 0.0819, 0.1238, 0.0907],
         [0.1206, 0.0845, 1.0000,  ..., 0.1333, 0.1198, 0.

In [60]:
params = {
    "dropout1": 0.1,
    "dropout2": 0.1,
    "n_drug": drug.shape[0],
    "n_cell": cell.shape[0],
    "n_gene": gene.shape[0],
    "hidden1": 256,
    "hidden2": 32,
    "hidden3": 128,
    "epochs": 100,
    "lr": 0.001,
    "heads": 2,
}

In [61]:
model, train_attention, val_attention = drGAT.train(data, params=params, device=device)

Using:  cpu
Epoch 1: Train Loss = 0.698, Val Loss = 0.682, Train Acc = 0.4957, Val Acc = 0.6032
Epoch 2: Train Loss = 0.6958, Val Loss = 0.6887, Train Acc = 0.5087, Val Acc = 0.5
Epoch 3: Train Loss = 0.6926, Val Loss = 0.6809, Train Acc = 0.5164, Val Acc = 0.6205
Epoch 4: Train Loss = 0.6922, Val Loss = 0.6954, Train Acc = 0.5225, Val Acc = 0.5041
Epoch 5: Train Loss = 0.6971, Val Loss = 0.6953, Train Acc = 0.5092, Val Acc = 0.5
Epoch 6: Train Loss = 0.6976, Val Loss = 0.6944, Train Acc = 0.5125, Val Acc = 0.4986
Epoch 7: Train Loss = 0.6964, Val Loss = 0.687, Train Acc = 0.5023, Val Acc = 0.5419
Epoch 8: Train Loss = 0.6919, Val Loss = 0.6776, Train Acc = 0.5293, Val Acc = 0.5713
Epoch 9: Train Loss = 0.6868, Val Loss = 0.6765, Train Acc = 0.5425, Val Acc = 0.5702
Epoch 10: Train Loss = 0.7042, Val Loss = 0.6749, Train Acc = 0.5068, Val Acc = 0.59
Epoch 11: Train Loss = 0.6818, Val Loss = 0.6774, Train Acc = 0.5589, Val Acc = 0.5789
Epoch 12: Train Loss = 0.6888, Val Loss = 0.6722, T

## Eval

In [62]:
test_drug = test_data["Drug"].values
test_cell = test_data["Cell"].values

test_labels = np.load("../CTRP_data/test_labels.npy")
test_labels = torch.tensor(test_labels).float()

In [63]:
test = [drug, cell, gene, edge_index, edge_attr, test_drug, test_cell, test_labels]

In [65]:
prob, res, test_attention = drGAT.eval(model, test)

In [66]:
res

,Accuracy,Precision,Recall,F1 Score,True Positive,True Negative,False Positive,False Negative
0,0.569352,0.539992,0.936427,0.684986,5347,1155,4555,363


In [67]:
test_attention

tensor([[0.0016, 0.0000, 0.0000,  ..., 0.0016, 0.0016, 0.0016],
        [0.0000, 0.0016, 0.0000,  ..., 0.0016, 0.0016, 0.0016],
        [0.0000, 0.0000, 0.0016,  ..., 0.0016, 0.0016, 0.0016],
        ...,
        [0.0016, 0.0016, 0.0016,  ..., 0.0016, 0.0000, 0.0000],
        [0.0016, 0.0016, 0.0016,  ..., 0.0000, 0.0016, 0.0000],
        [0.0016, 0.0016, 0.0016,  ..., 0.0000, 0.0000, 0.0016]])